In [13]:
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 20000)
import warnings
warnings.filterwarnings('ignore')
# data = pd.read_excel(r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260202_창조교육과\02_활용자료\\2024 외래관광객조사_Data.xlsx", engine='openpyxl')
# output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260202_창조교육과\\"
data = pd.read_spss(r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260202_창조교육과\02_활용자료\\2024 외래관광객조사_Data.SAV")
data = data[['Q9_2a13','Q11','Q8_1a1']]
data.loc[(~data['Q9_2a13'].isna()) & (data['Q11'].str.contains('매우 만족|만족'))]['Q8_1a1'].value_counts().to_frame().T

Q8_1a1,식도락(음식/미식) 관광,고궁/역사 유적지 방문,자연경관 감상,쇼핑,"전통문화체험 (한복체험, 태권도, 한국음식 만들기 등)",오락(놀이공원 및 테마파크 방문 등),"박물관, 전시관 관람","비즈니스 전문활동(업무회의, 산업시설 시찰 등)",K-POP/한류스타 관련 공연장 및 드라마/영화 촬영지 방문,휴양/휴식(웰니스),유흥(나이트라이프 활동 및 카지노 등),"뷰티/미용 관광(마사지샵, 헤어샵, 네일케어 등 방문)","연수, 교육, 연구","치료/건강검진(피부과, 성형외과, 내외과 등 입원, 수술 등)","스포츠/레포츠 참가 (스키, 수영, 골프, 캠핑, 등반, 자전거, 래프팅 등)",지역 축제 참여,"연극, 뮤지컬, 발레 등 공연 관람","스포츠/레포츠 관람 (야구, 축구, 농구 등)","국제회의(컨벤션, 국제회의, 전시 등)",기타
count,510,507,477,315,205,148,139,116,95,86,78,74,48,35,25,21,15,14,11,3


#### 거주국별 관광수용태세 개선 영역 분석

In [2]:
data3 = data.loc[~data['Q9_2a13'].isna(), 
                 ['Q11','Q12a01','Q12a02','Q12a03','Q12a04','Q12a05','Q12a06','Q12a07','Q12a08','Q12a09','Q12a10',
                  'Q12a11','Q12a12','Q12a13','Q12a14','Q12a15','Q12a16','Q12a17','Q12a18','Q12a19','Q12a20',
                  'Q12a21','Q12a22','Q12a23','D_NAT']].copy()

nat_dict = {"code": ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19','20','97'],
            "name": ['중국','일본','대만','미국','홍콩','태국','베트남','말레시아','필리핀','싱가포르','러시아','중동','인도네시아',
                     '캐나다','호주','영국','몽골','독일','프랑스', '인도','기타']}    
nat_code = pd.DataFrame(nat_dict)
data3['D_NAT'] = data3['D_NAT'].astype(int).astype(str)
data3 = pd.merge(data3, nat_code, "left", left_on="D_NAT", right_on="code").drop(columns=['D_NAT','code'])
data3.columns = ['전반적 만족도','식도락','쇼핑','자연경관 감상','휴양/휴식','고궁/역사 유적지 방문','전통문화체험','박물관,전시관 관람',
                 'K-POP/한류스타 관련 공연장 및 드라마/영화 촬영지 방문','연극,뮤지컬,발레 등 공연 관람',
                 '지역축제참여','유흥','오락','뷰티/미용 관광','치료/건강검진','스포츠/레포츠 관람',
                 '스포츠/레포츠 참가','출입국 절차','언어소통','치안','여행 경비','숙박 시설','대중교통/교통','길찾기','거주국']

cols_to_convert = data3.columns.drop('거주국')
data3[cols_to_convert] = data3[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# 항목 분류 및 정렬 맵 설정
master_item_list = list(data3.columns[1:24])
activities_cols = master_item_list[:16]    # Q12a01~16
infrastructure_cols = master_item_list[16:] # Q12a17~23
item_order_map = {item: i for i, item in enumerate(master_item_list)}

def analyze_ipa_by_category_report_style(data, items, category_name):
    results = []
    for country, group in data.groupby('거주국'):
        available_items = [c for c in items if c in group.columns]
        if not available_items: continue
        
        # 1) 만족도(Performance) : 평균 산출
        performance = group[available_items].mean(numeric_only=True)
        
        # 2) 상관분석 실시 (보고서 정의 1단계)
        # 각 항목 만족도와 전반적 만족도 간의 상관계수(r) 도출
        corr_series = group[available_items].apply(lambda x: x.corr(group['전반적 만족도']))
        
        # 3) 중요도(Importance) : 상관계수 합에 대한 비중 산출 (보고서 정의 2단계)
        # 양의 상관관계가 있는 항목들의 합을 기준으로 비중(%) 계산
        positive_corr = corr_series[corr_series > 0]
        total_corr = positive_corr.sum()
        
        if total_corr > 0:
            importance = (corr_series / total_corr) * 100
        else:
            importance = corr_series # 합산 불가능한 경우 원본값 유지
            
        ipa = pd.DataFrame({'항목': available_items, '만족도': performance, '중요도': importance}).dropna(subset=['중요도'])
        
        if len(ipa) == 0: continue
        
        # 기준선 설정: 만족도는 3.0, 중요도는 비중의 평균
        v_line = ipa['중요도'].mean()
        h_line = 3.0
        
        def get_quadrant(row):
            if row['중요도'] >= v_line and row['만족도'] >= h_line: return '적극 권장 영역'
            elif row['중요도'] < v_line and row['만족도'] >= h_line: return '현상 유지 영역'
            elif row['중요도'] < v_line and row['만족도'] < h_line: return '개선 검토 영역'
            else: return '최우선 개선 영역'
        
        ipa['영역'] = ipa.apply(get_quadrant, axis=1)
        
        # 영역별 결과 정리 (불렛포인트 및 정렬 적용)
        summary_dict = {'거주국': country, '구분': category_name}
        quads = ['적극 권장 영역', '현상 유지 영역', '개선 검토 영역', '최우선 개선 영역']
        
        for quad in quads:
            quad_items = ipa[ipa['영역'] == quad]['항목'].tolist()
            quad_items.sort(key=lambda x: item_order_map[x]) # 원문 순서 정렬
            summary_dict[quad] = "\n".join(["• " + itm for itm in quad_items]) if quad_items else "-"
            
        results.append(pd.DataFrame([summary_dict]))
    return results

# 2. 분석 실행 (관광활동 -> 관광인프라 순서 유지)
final_list = analyze_ipa_by_category_report_style(data3, activities_cols, '관광활동') + \
             analyze_ipa_by_category_report_style(data3, infrastructure_cols, '관광인프라')

# 3. 결과 통합 및 엑셀 저장
final_df = pd.concat(final_list).set_index(['거주국', '구분'])
final_df = final_df[['적극 권장 영역', '현상 유지 영역', '개선 검토 영역', '최우선 개선 영역']]

# 저장 (줄바꿈 서식 적용을 위해 xlsxwriter 엔진 사용)
writer = pd.ExcelWriter(output_path + '분석결과_거주국별개선영역분석_최종.xlsx', engine='xlsxwriter')
final_df.to_excel(writer, sheet_name='Sheet1')

# 엑셀 서식 강제 적용 (줄바꿈 및 정렬)
workbook = writer.book
worksheet = writer.sheets['Sheet1']
wrap_fmt = workbook.add_format({'text_wrap': True, 'align': 'left', 'valign': 'vcenter'})
index_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter'})

worksheet.set_column('A:B', 15, index_fmt) # 거주국, 구분
worksheet.set_column('C:F', 45, wrap_fmt)  # 영역별 결과 (줄바꿈 적용)
writer.close()

print("보고서 기준이 반영된 분석이 완료되었습니다.")


보고서 기준이 반영된 분석이 완료되었습니다.
